# Car Price Prediction using Linear Regression

Research AI Task 6

The aim is to predict car prices using linear regression without using scikit-learn.

## 1. Import libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Load the dataset

In [2]:
df = pd.read_csv("CarPrice_Assignment.csv")
df.head()

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,carheight,curbweight,enginetype,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,four,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,five,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


## 3. Basic inspection

In [3]:
print("Shape:", df.shape)
print("\nMissing values:", df.isnull().sum().sum())
print("\nData types:")
print(df.dtypes)

Shape: (205, 26)

Missing values: 0

Data types:
car_ID                int64
symboling             int64
CarName              object
fueltype             object
aspiration           object
doornumber           object
carbody              object
drivewheel           object
enginelocation       object
wheelbase           float64
carlength           float64
carwidth            float64
carheight           float64
curbweight            int64
enginetype           object
cylindernumber       object
enginesize            int64
fuelsystem           object
boreratio           float64
stroke              float64
compressionratio    float64
horsepower            int64
peakrpm               int64
citympg               int64
highwaympg            int64
price               float64
dtype: object


## 4. Select features

In [4]:
numeric_features = [
    col for col in df.select_dtypes(include=np.number).columns
    if col not in ["price", "car_ID"]
]

categorical_features = [
    col for col in df.select_dtypes(include="object").columns
    if col != "CarName"
]

X_df = pd.get_dummies(
    df[numeric_features + categorical_features],
    drop_first=True,
    dtype=float
)

X = X_df.to_numpy()
y = df["price"].to_numpy()

print("Number of features:", X.shape[1])

Number of features: 43


## 5. Train-test split

In [5]:
rng = np.random.default_rng(42)
indices = rng.permutation(len(X))

split = int(0.8 * len(X))
train_idx = indices[:split]
test_idx = indices[split:]

X_train = X[train_idx]
X_test = X[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 164
Testing samples: 41


## 6. Feature scaling

In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

# Remove constant columns so standardisation does not divide by zero
keep = std > 0
X_train = (X_train[:, keep] - mean[keep]) / std[keep]
X_test = (X_test[:, keep] - mean[keep]) / std[keep]

# Add a bias column for the intercept
X_train = np.c_[np.ones(len(X_train)), X_train]
X_test = np.c_[np.ones(len(X_test)), X_test]


## 7. Linear regression using gradient descent

In [ ]:
theta = np.zeros(X_train.shape[1])

learning_rate = 0.01
iterations = 20000
cost_history = []

for i in range(iterations):
    error = X_train @ theta - y_train
    gradient = (X_train.T @ error) / len(X_train)
    theta -= learning_rate * gradient

    if i % 100 == 0:
        cost = np.mean((X_train @ theta - y_train) ** 2) / 2
        cost_history.append(cost)


## 8. Cost function during training

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(np.arange(len(cost_history)) * 100, cost_history)
plt.xlabel("Training iterations")
plt.ylabel("Half mean squared error")
plt.title("Training Cost During Gradient Descent")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Predictions

In [ ]:
train_predictions = X_train @ theta
test_predictions = X_test @ theta

print("First 10 actual test prices:")
print(y_test[:10])

print("\\nFirst 10 predicted test prices:")
print(test_predictions[:10])


## 10. Model evaluation

In [ ]:
def regression_metrics(actual, predicted, number_of_features=None):
    errors = predicted - actual
    mse = np.mean(errors ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))
    median_ae = np.median(np.abs(errors))
    nonzero = actual != 0
    mape = np.mean(np.abs(errors[nonzero] / actual[nonzero])) * 100
    ss_res = np.sum((actual - predicted) ** 2)
    ss_tot = np.sum((actual - actual.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

    results = {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "Median Absolute Error": median_ae,
        "MAPE (%)": mape,
        "R-squared": r2
    }

    if number_of_features is not None and len(actual) > number_of_features + 1:
        adjusted_r2 = 1 - (1 - r2) * (len(actual) - 1) / (len(actual) - number_of_features - 1)
        results["Adjusted R-squared"] = adjusted_r2

    return results

train_metrics = regression_metrics(
    y_train, train_predictions, number_of_features=X_train.shape[1] - 1
)
test_metrics = regression_metrics(
    y_test, test_predictions, number_of_features=X_test.shape[1] - 1
)

print("TRAINING METRICS")
for metric, value in train_metrics.items():
    print(f"{metric}: {value:.3f}")

print("\\nTEST METRICS")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.3f}")


## 11. Actual vs predicted prices

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, test_predictions, alpha=0.8)

lower = min(y_test.min(), test_predictions.min())
upper = max(y_test.max(), test_predictions.max())
plt.plot([lower, upper], [lower, upper], linestyle="--", label="Perfect predictions")

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Car Prices")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

residuals = y_test - test_predictions
plt.figure(figsize=(7, 5))
plt.scatter(test_predictions, residuals, alpha=0.8)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 12. Result and interpretation

The model was trained using an 80/20 train-test split and implemented with NumPy rather than scikit-learn.

The evaluation cell prints training and test metrics, including MSE, RMSE, MAE, median absolute error, MAPE, R-squared, and adjusted R-squared where applicable.

The training-cost curve shows how the optimisation objective changes during gradient descent. A decreasing training cost alone does not prove overfitting; compare training and test metrics to assess generalisation.

The actual-versus-predicted plot includes a diagonal reference line for perfect predictions. The residual plot helps reveal systematic errors or patterns the model may not capture.

Report the values printed when the notebook is run with the supplied dataset. Do not replace the measured results with a desired RMSE value.
